In [83]:
from pathlib import Path

%load_ext autoreload
%autoreload 2

import datetime as dt
import pandas as pd
import numpy as np

from wood_charts import load_theme
from wood_charts.charts import line_chart, bar_chart
theme = load_theme("notebook")

from synthetic_website_analytics.data import DatabaseConnector


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [84]:
project_root = Path.cwd().parent
connector = DatabaseConnector.from_env(project_root / ".env")

In [85]:
query_path = project_root / "sql" / "performance" / "daily_performance.sql"
daily_performance = connector.execute_sql(query_path)

daily_performance["date_day"] = pd.to_datetime(
    daily_performance["date_day"]
)
daily_performance

,date_day,session_count,visitor_count,event_count,page_view_count,product_view_count,add_to_cart_count,begin_checkout_count,purchase_count,product_view_session_count,...,order_count,total_order_value,total_items_purchased,session_to_product_view_rate,product_view_to_cart_rate,cart_to_checkout_rate,checkout_to_purchase_rate,session_conversion_rate,average_order_value,revenue_per_session
0,2026-01-01,443,286,1976.0,1295.0,190.0,194.0,100.0,82.0,128,...,82.0,18365.02,273.0,0.288939,1.187500,0.657895,0.820000,0.185102,223.963659,41.456027
1,2026-01-02,460,331,2117.0,1378.0,197.0,227.0,116.0,95.0,138,...,95.0,22729.27,346.0,0.300000,1.253623,0.670520,0.818966,0.206522,239.255474,49.411457
2,2026-01-03,237,186,1188.0,748.0,128.0,132.0,71.0,59.0,85,...,59.0,13264.73,226.0,0.358650,1.129412,0.739583,0.830986,0.248945,224.825932,55.969325
3,2026-01-04,156,138,771.0,498.0,76.0,81.0,43.0,39.0,54,...,39.0,9132.29,147.0,0.346154,1.092593,0.728814,0.906977,0.250000,234.161282,58.540321
4,2026-01-05,428,328,1927.0,1249.0,185.0,201.0,101.0,85.0,133,...,85.0,19119.38,287.0,0.310748,1.105263,0.687075,0.841584,0.198598,224.933882,44.671449
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,2026-12-27,167,144,740.0,490.0,70.0,62.0,35.0,30.0,54,...,30.0,6947.78,102.0,0.323353,0.944444,0.686275,0.857143,0.179641,231.592667,41.603473
361,2026-12-28,457,375,1996.0,1350.0,180.0,205.0,97.0,78.0,122,...,78.0,19563.24,258.0,0.266958,1.221311,0.651007,0.804124,0.170678,250.810769,42.807965
362,2026-12-29,499,417,2178.0,1416.0,206.0,223.0,121.0,100.0,150,...,100.0,23091.86,347.0,0.300601,1.186667,0.679775,0.826446,0.200401,230.918600,46.276273
363,2026-12-30,477,402,2045.0,1350.0,192.0,205.0,111.0,90.0,136,...,90.0,21663.46,315.0,0.285115,1.132353,0.720779,0.810811,0.188679,240.705111,45.416059


In [86]:
daily_performance = daily_performance.assign(
    session_conversion_rate=(
        daily_performance["purchase_session_count"]
        / daily_performance["session_count"]
    ),
    average_order_value=(
        daily_performance["total_order_value"]
        / daily_performance["order_count"]
    ),
    revenue_per_session=(
        daily_performance["total_order_value"]
        / daily_performance["session_count"]
    ),
)

daily_performance["sessions_7d_avg"] = (
    daily_performance["session_count"]
    .rolling(7, min_periods=1)
    .mean()
)

daily_performance["conversion_rate_7d_avg"] = (
    daily_performance["session_conversion_rate"]
    .rolling(7, min_periods=1)
    .mean()
)

daily_performance["revenue_7d_avg"] = (
    daily_performance["total_order_value"]
    .rolling(7, min_periods=1)
    .mean()
)

In [87]:
figure = line_chart(
    daily_performance,
    x="date_day",
    y="session_count",
    theme=theme,
    title="Website Traffic Shows Strong Weekly Seasonality",
    subtitle="Daily sessions, January 2026–January 2027",
    x_axis_title="Date",
    y_axis_title="Sessions",
    source="Synthetic Website Data, marts.fct_website_daily_metrics",
)

figure.show()

In [88]:
daily_performance["day_of_week"] = (
    daily_performance["date_day"]
    .dt.day_name()
)

weekday_performance = (
    daily_performance
    .groupby("day_of_week", as_index=False)
    .agg(
        avg_sessions=("session_count", "mean"),
        avg_visitors=("visitor_count", "mean"),
    )
)

day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

daily_performance["day_type"] = np.where(
    daily_performance["date_day"].dt.dayofweek < 5,
    "Weekday",
    "Weekend",
)

day_type_performance = (
    daily_performance
    .groupby("day_type", as_index=False)
    .agg(
        avg_sessions=("session_count", "mean"),
        avg_visitors=("visitor_count", "mean"),
    )
)

weekday_avg = day_type_performance.loc[
    day_type_performance["day_type"] == "Weekday",
    "avg_sessions",
].iloc[0]

weekend_avg = day_type_performance.loc[
    day_type_performance["day_type"] == "Weekend",
    "avg_sessions",
].iloc[0]

weekend_decline = 1 - (weekend_avg / weekday_avg)

weekday_performance["day_of_week"] = pd.Categorical(
    weekday_performance["day_of_week"],
    categories=day_order,
    ordered=True,
)

weekday_performance = weekday_performance.sort_values("day_of_week")

figure = bar_chart(
    weekday_performance,
    x="day_of_week",
    y="avg_sessions",
    theme=theme,
    title=f"Weekend Traffic Is {weekend_decline:.0%} Lower vs. Weekdays",
    subtitle="Weekend sessions compared with weekday average",
    x_axis_title="Day of Week",
    y_axis_title="Average Sessions",
    focus=["Saturday", "Sunday"],
    source="Synthetic Website Data, marts.fct_website_daily_metrics"
)
figure.show()

In [96]:
figure = line_chart(
    daily_performance,
    x="date_day",
    y="sessions_7d_avg",
    theme=theme,
    title="Spring Traffic Peak Coincides With Campaign Activity",
    subtitle="7-day rolling average of daily sessions",
    x_axis_title="Date",
    y_axis_title="Sessions",
    event_bands=[
    {
        "start": "2026-03-01",
        "end": "2026-04-15",
        "label": "Spring campaign",
    }
],
    source="Synthetic Website Data, marts.fct_website_daily_metrics",
)

figure.show()

In [ ]:
# connector.dispose()